# Order Data Generation

This notebook generates realistic order lifecycle data with multiple event types (NEW, TRADE, CANCEL) and various order completion patterns.

## Features:
- Log-normal distributed time intervals for realistic timing
- Multiple lifecycle patterns: FULL_FILL, PARTIAL_FILL, CANCEL, PARTIAL_CANCEL
- Proper timestamp sequencing across events
- Configurable parameters for order generation

In [72]:
# Import dependencies
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

## Configuration

Set random seeds for reproducibility and configure generation parameters.

In [73]:
# Set random seeds for reproducibility
np.random.seed(42)
random.seed(42)

## Main Generation Function

Single comprehensive function that generates order data with all lifecycle patterns.

In [74]:
def generate_orders(n_orders=100, start_order_id=1000, start_time=None, 
                   base_price=101.35, price_std=0.1, n_traders=20):
    """
    Generate multiple orders with complete lifecycles including all event types (NEW, TRADE, CANCEL).
    
    Parameters:
    -----------
    n_orders : int
        Number of orders to generate
    start_order_id : int
        Starting order ID
    start_time : datetime, optional
        Starting timestamp (defaults to Jan 18, 2026 9:30 AM)
    base_price : float
        Base price for order pricing
    price_std : float
        Standard deviation for price distribution
    n_traders : int
        Number of unique traders
    
    Returns:
    --------
    pd.DataFrame
        DataFrame containing all order events, sorted by timestamp
    """
    # Helper function for log-normal time distribution
    def lognormal_time_ms(min_ms, max_ms):
        mean = np.log((min_ms + max_ms) / 2)
        sigma = np.log(max_ms / min_ms) / 3.0
        value = np.random.lognormal(mean, sigma)
        return int(max(min_ms, min(value, max_ms)))
    
    if start_time is None:
        start_time = datetime(2026, 1, 18, 9, 30, 0)
    
    orders = []
    order_id = start_order_id
    current_time = start_time
    
    # Available quantities and lifecycles
    available_quantities = [100, 150, 200, 250, 300]
    lifecycles = ["FULL_FILL", "PARTIAL_FILL", "CANCEL", "PARTIAL_CANCEL"]
    
    for _ in range(n_orders):
        order_id += 1
        trader_id = f"T{random.randint(1, n_traders):02d}"
        side = random.choice(["Buy", "Sell"])
        price = round(np.random.normal(base_price, price_std), 2)
        quantity = random.choice(available_quantities)
        order_type = "Limit"
        lifecycle = random.choice(lifecycles)
        
        # NEW event
        orders.append({
            "order_id": order_id,
            "trader_id": trader_id,
            "side": side,
            "price": price,
            "quantity": quantity,
            "remaining": quantity,
            "event_type": "NEW",
            "order_type": order_type,
            "timestamp": current_time,
            "status": "Open"
        })
        current_time += timedelta(milliseconds=lognormal_time_ms(50, 200))
        
        remaining = quantity
        
        # Process lifecycle patterns
        if lifecycle in ["FULL_FILL", "PARTIAL_FILL", "PARTIAL_CANCEL"]:
            # First trade
            trade_size = random.randint(int(0.3 * quantity), int(0.7 * quantity))
            remaining -= trade_size
            
            orders.append({
                "order_id": order_id,
                "trader_id": trader_id,
                "side": side,
                "price": price,
                "quantity": quantity,
                "remaining": remaining,
                "event_type": "TRADE",
                "order_type": order_type,
                "timestamp": current_time,
                "status": "Partially filled" if remaining > 0 else "Filled"
            })
            current_time += timedelta(milliseconds=lognormal_time_ms(50, 200))
        
        # Handle different lifecycle endings
        if lifecycle == "PARTIAL_FILL":
            # Final trade to fill remaining
            orders.append({
                "order_id": order_id,
                "trader_id": trader_id,
                "side": side,
                "price": price,
                "quantity": quantity,
                "remaining": 0,
                "event_type": "TRADE",
                "order_type": order_type,
                "timestamp": current_time,
                "status": "Filled"
            })
            current_time += timedelta(milliseconds=lognormal_time_ms(50, 200))
        
        elif lifecycle == "FULL_FILL":
            # Second trade to fill remaining
            current_time += timedelta(milliseconds=lognormal_time_ms(50, 200))
            orders.append({
                "order_id": order_id,
                "trader_id": trader_id,
                "side": side,
                "price": price,
                "quantity": quantity,
                "remaining": 0,
                "event_type": "TRADE",
                "order_type": order_type,
                "timestamp": current_time,
                "status": "Filled"
            })
        
        elif lifecycle == "PARTIAL_CANCEL":
            # Cancel after partial fill
            orders.append({
                "order_id": order_id,
                "trader_id": trader_id,
                "side": side,
                "price": price,
                "quantity": quantity,
                "remaining": remaining,
                "event_type": "CANCEL",
                "order_type": order_type,
                "timestamp": current_time,
                "status": "Cancelled"
            })
            current_time += timedelta(milliseconds=lognormal_time_ms(50, 200))
        
        elif lifecycle == "CANCEL":
            # Cancel without any trades
            orders.append({
                "order_id": order_id,
                "trader_id": trader_id,
                "side": side,
                "price": price,
                "quantity": quantity,
                "remaining": remaining,
                "event_type": "CANCEL",
                "order_type": order_type,
                "timestamp": current_time,
                "status": "Cancelled"
            })
            current_time += timedelta(milliseconds=lognormal_time_ms(50, 200))
        
        # Increment time between orders
        current_time += timedelta(milliseconds=lognormal_time_ms(100, 400))
    
    # Create DataFrame and sort by timestamp
    df = pd.DataFrame(orders)
    df = df.sort_values("timestamp").reset_index(drop=True)
    
    return df

## Market Abuse Data Generation Function

Generate order data that exhibits market abuse behaviors including:
- Cancel dominance (high cancel ratio)
- Extremely short order lifetimes
- Abnormally high message rates
- Size deception (large orders, minimal execution)

In [75]:
def generate_abuse_orders(
    n_orders=100,
    start_order_id=2000,
    start_time=None,
    base_price=101.35,

    # Microstructure realism
    tick_size=0.01,
    lot_size=10,
    price_std_ticks=2.0,          # std dev in ticks (e.g., 2 ticks)
    n_traders=3,                  # abuse is concentrated (2–4 common)

    # Abuse profile
    cancel_ratio_target=0.95,     # abuse-like (0.90–0.99)
    min_lifetime_us=50,           # very fast cancels possible
    max_lifetime_us=50_000,       # heavy tail up to 50ms

    # Message dynamics (bursty)
    baseline_iat_us=500,          # typical inter-arrival outside bursts
    burst_prob=0.15,              # chance to enter burst regime
    burst_len_range=(10, 60),     # burst length in orders
    burst_iat_us_range=(5, 50),   # inter-arrival during bursts (us)

    # Size deception
    large_size_prob=0.35,         # fraction of large “spoof” orders
    large_size_mult=10,           # large order multiplier
    tiny_fill_frac_range=(0.001, 0.02),  # 0.1%–2% executed when traded


):
    """
    Generate an order-event log exhibiting market abuse behaviors (quote stuffing / spoof-like patterns).

    Output schema (event log):
    - order_id, trader_id, side, price, quantity, remaining, event_type, order_type, timestamp, status

    Key characteristics:
    - High cancel dominance (target cancel ratio)
    - Short lifetimes (mixture: microseconds + milliseconds tail)
    - Bursty message rate (micro-bursts)
    - Size deception (large posted size, tiny execution when trades happen)
    """


    if start_time is None:
        start_time = datetime(2026, 1, 18, 9, 30, 0)

    # --- helpers (tick/lot rounding) ---
    def round_to_tick(p: float) -> float:
        return float(np.round(np.round(p / tick_size) * tick_size, 10))

    def round_to_lot(q: float) -> int:
        q2 = int(np.round(q / lot_size) * lot_size)
        return max(lot_size, q2)

    # --- sample “session-level” parameters with small noise (more realistic than ±5%) ---
    # Base price: a few bps of drift/noise, not percent-level
    effective_base_price = float(np.random.normal(base_price, max(1e-9, base_price * 0.0002)))  # ~2 bps
    effective_base_price = round_to_tick(effective_base_price)

    # Traders: concentrated
    effective_n_traders = int(np.clip(np.random.poisson(lam=n_traders), 2, 8))

    # Cancel ratio: tight around target (less random chaos)
    # Scale up alpha/beta to reduce variance
    k = 60
    a = max(1e-6, cancel_ratio_target * k)
    b = max(1e-6, (1 - cancel_ratio_target) * k)
    effective_cancel_ratio = float(np.clip(np.random.beta(a, b), 0.85, 0.995))

    # Lifetimes: mixture distribution (most microseconds, some ms tail)
    def sample_lifetime_us() -> int:
        # 80%: very short, 20%: ms tail
        if np.random.rand() < 0.80:
            # lognormal concentrated in tens–thousands of µs
            x = np.random.lognormal(mean=np.log(max(min_lifetime_us, 20)), sigma=0.8)
        else:
            # ms tail
            x = np.random.lognormal(mean=np.log(max(5_000, min_lifetime_us)), sigma=1.0)
        return int(np.clip(x, min_lifetime_us, max_lifetime_us))

    # Inter-arrival times: bursty mixture
    burst_remaining = 0

    def sample_iat_us() -> int:
        nonlocal burst_remaining
        if burst_remaining > 0:
            burst_remaining -= 1
            return int(np.random.randint(burst_iat_us_range[0], burst_iat_us_range[1] + 1))

        # maybe start a burst
        if np.random.rand() < burst_prob:
            burst_remaining = int(np.random.randint(burst_len_range[0], burst_len_range[1] + 1))
            return int(np.random.randint(burst_iat_us_range[0], burst_iat_us_range[1] + 1))

        # outside bursts: exponential around baseline
        x = np.random.exponential(scale=baseline_iat_us)
        return int(max(1, x))

    # Price noise in ticks (more interpretable)
    price_std = price_std_ticks * tick_size

    # Realistic “lumpy” sizes + tail + deception
    typical_lots = [50, 100, 200, 500, 1000]  # units (pre-lot rounding)
    order_type = "Limit"

    abuse_traders = [f"ABUSE_T{tid:02d}" for tid in range(1, effective_n_traders + 1)]

    # Decide outcomes per order to hit cancel ratio approximately:
    # - CANCEL_ONLY: NEW->CANCEL
    # - TRADE_CANCEL: NEW->TRADE->CANCEL (tiny fill)
    # - TRADE_FILL:   NEW->TRADE->(maybe TRADE)->FILLED (rare)
    #
    # We pick probabilities so that most orders end in cancel.
    p_trade_any = max(0.01, 1.0 - effective_cancel_ratio)  # small
    p_trade_fill = 0.10 * p_trade_any                      # very rare fills
    p_trade_cancel = p_trade_any - p_trade_fill
    p_cancel_only = 1.0 - p_trade_any

    # Generate events
    events = []
    order_id = start_order_id
    current_time = start_time

    for i in range(n_orders):
        order_id += 1
        trader_id = random.choice(abuse_traders)
        side = random.choice(["Buy", "Sell"])

        # Size: mixture typical + large spoof sizes
        base_q = random.choice(typical_lots)
        if np.random.rand() < large_size_prob:
            quantity = base_q * large_size_mult
        else:
            # add some natural variation sometimes
            if np.random.rand() < 0.25:
                quantity = base_q * int(np.random.choice([1, 2, 3]))
            else:
                quantity = base_q
        quantity = round_to_lot(quantity)

        # Price: around base, but rounded to tick
        price = round_to_tick(np.random.normal(effective_base_price, price_std))

        # NEW
        new_time = current_time
        events.append({
            "order_id": order_id,
            "trader_id": trader_id,
            "side": side,
            "price": price,
            "quantity": quantity,
            "remaining": quantity,
            "event_type": "NEW",
            "order_type": order_type,
            "timestamp": new_time,
            "status": "Open"
        })

        # Outcome draw
        u = np.random.rand()
        if u < p_cancel_only:
            # NEW -> CANCEL
            cancel_time = new_time + timedelta(microseconds=sample_lifetime_us())
            events.append({
                "order_id": order_id,
                "trader_id": trader_id,
                "side": side,
                "price": price,
                "quantity": quantity,
                "remaining": quantity,
                "event_type": "CANCEL",
                "order_type": order_type,
                "timestamp": cancel_time,
                "status": "Cancelled"
            })
            current_time = cancel_time

        elif u < p_cancel_only + p_trade_cancel:
            # NEW -> TRADE (tiny) -> CANCEL (rest)
            fill_frac = float(np.random.uniform(*tiny_fill_frac_range))
            trade_size = max(1, int(quantity * fill_frac))
            trade_size = min(trade_size, quantity - 1)  # ensure remaining > 0
            remaining = quantity - trade_size

            trade_time = new_time + timedelta(microseconds=max(1, int(sample_lifetime_us() * 0.2)))
            events.append({
                "order_id": order_id,
                "trader_id": trader_id,
                "side": side,
                "price": price,
                "quantity": quantity,
                "remaining": remaining,
                "event_type": "TRADE",
                "order_type": order_type,
                "timestamp": trade_time,
                "status": "Partially filled"
            })

            cancel_time = trade_time + timedelta(microseconds=max(1, int(sample_lifetime_us() * 0.2)))
            events.append({
                "order_id": order_id,
                "trader_id": trader_id,
                "side": side,
                "price": price,
                "quantity": quantity,
                "remaining": remaining,
                "event_type": "CANCEL",
                "order_type": order_type,
                "timestamp": cancel_time,
                "status": "Cancelled"
            })
            current_time = cancel_time

        else:
            # NEW -> TRADE(s) -> FILLED (rare)
            # First partial trade
            first_trade = max(1, int(quantity * float(np.random.uniform(0.2, 0.7))))
            remaining = quantity - first_trade

            trade_time1 = new_time + timedelta(microseconds=max(1, int(sample_lifetime_us() * 0.3)))
            events.append({
                "order_id": order_id,
                "trader_id": trader_id,
                "side": side,
                "price": price,
                "quantity": quantity,
                "remaining": remaining,
                "event_type": "TRADE",
                "order_type": order_type,
                "timestamp": trade_time1,
                "status": "Partially filled" if remaining > 0 else "Filled"
            })

            if remaining > 0:
                # Final trade
                trade_time2 = trade_time1 + timedelta(microseconds=max(1, int(sample_lifetime_us() * 0.6)))
                events.append({
                    "order_id": order_id,
                    "trader_id": trader_id,
                    "side": side,
                    "price": price,
                    "quantity": quantity,
                    "remaining": 0,
                    "event_type": "TRADE",
                    "order_type": order_type,
                    "timestamp": trade_time2,
                    "status": "Filled"
                })
                current_time = trade_time2
            else:
                current_time = trade_time1

        # Next order arrival (bursty)
        current_time = current_time + timedelta(microseconds=sample_iat_us())

    df = pd.DataFrame(events).sort_values("timestamp").reset_index(drop=True)
    return df

## Try abuse function

In [76]:
# --- Example call with realistic parameters (abuse-y but plausible) ---
df_abuse = generate_abuse_orders(
    n_orders=100,
    start_order_id=2000,
    start_time=datetime(2026, 1, 6, 9, 30, 0),
    base_price=101.35,

    tick_size=0.01,
    lot_size=10,
    price_std_ticks=1.5,
    n_traders=3,

    cancel_ratio_target=0.96,
    min_lifetime_us=50,
    max_lifetime_us=50_000,

    baseline_iat_us=600,
    burst_prob=0.18,
    burst_len_range=(15, 80),
    burst_iat_us_range=(5, 40),

    large_size_prob=0.40,
    large_size_mult=20,
    tiny_fill_frac_range=(0.001, 0.01),
)


In [77]:
display(df_abuse)

,order_id,trader_id,side,price,quantity,remaining,event_type,order_type,timestamp,status
0,2001,ABUSE_T03,Buy,101.38,1000,1000,NEW,Limit,2026-01-06 09:30:00.000000,Open
1,2001,ABUSE_T03,Buy,101.38,1000,992,TRADE,Limit,2026-01-06 09:30:00.000010,Partially filled
2,2001,ABUSE_T03,Buy,101.38,1000,992,CANCEL,Limit,2026-01-06 09:30:00.000020,Cancelled
3,2002,ABUSE_T03,Sell,101.34,2000,2000,NEW,Limit,2026-01-06 09:30:00.000359,Open
4,2002,ABUSE_T03,Sell,101.34,2000,2000,CANCEL,Limit,2026-01-06 09:30:00.000423,Cancelled
...,...,...,...,...,...,...,...,...,...,...
201,2098,ABUSE_T02,Buy,101.37,1000,1000,CANCEL,Limit,2026-01-06 09:30:00.221328,Cancelled
202,2099,ABUSE_T01,Buy,101.37,20000,20000,NEW,Limit,2026-01-06 09:30:00.221361,Open
203,2099,ABUSE_T01,Buy,101.37,20000,20000,CANCEL,Limit,2026-01-06 09:30:00.221411,Cancelled
204,2100,ABUSE_T01,Sell,101.36,20000,20000,NEW,Limit,2026-01-06 09:30:00.221420,Open


## Generate Combined Normal and Abuse Data

Generate both normal trading data and market abuse data, then merge them together.

In [78]:
# Configuration - Realistic Trading Behavior Parameters

# ============================================================================
# MARKET PARAMETERS (Realistic Equity/ETF Market)
# ============================================================================
BASE_PRICE = 101.35              # Typical mid-cap stock price
PRICE_STD = 0.15                 # ~0.15% volatility (realistic intraday)
TICK_SIZE = 0.01                 # Standard tick size for most US equities
LOT_SIZE = 10                    # Standard lot size

# ============================================================================
# NORMAL TRADING BEHAVIOR
# ============================================================================
N_NORMAL_ORDERS = 100000
N_NORMAL_TRADERS = 1000         # Large diverse pool (realistic for active market)

# Normal order characteristics:
# - Cancel ratio: 30-50% (realistic for limit orders)
# - Average lifetime: 100ms - 5 seconds
# - Message rate: 50-200 msgs/sec (varies by market activity)
# - Size ratio: 40-70% (most orders get filled or partially filled)

# ============================================================================
# ABUSE TRADING BEHAVIOR (Quote Stuffing / Spoofing)
# ============================================================================
N_ABUSE_ORDERS = 30000
N_ABUSE_TRADERS = 8              # Small concentrated group (typical abuse pattern)

# Abuse order characteristics:
# - Cancel ratio: 90-98% (extremely high - hallmark of abuse)
# - Average lifetime: 50μs - 5ms (extremely fast cancels)
# - Message rate: 500-2000 msgs/sec during bursts (10x normal)
# - Size ratio: 0.1-2% (large posted size, tiny execution = spoofing)

# Abuse-specific parameters:
ABUSE_CANCEL_RATIO_TARGET = 0.96      # 96% cancel rate (very suspicious)
ABUSE_MIN_LIFETIME_US = 50            # Can cancel in microseconds
ABUSE_MAX_LIFETIME_US = 5000           # Max 5ms lifetime (very short)

# Burst behavior (quote stuffing pattern):
ABUSE_BASELINE_IAT_US = 600            # ~1.7k msgs/sec baseline
ABUSE_BURST_PROB = 0.20                # 20% chance of entering burst
ABUSE_BURST_LEN_RANGE = (20, 100)      # Bursts of 20-100 orders
ABUSE_BURST_IAT_US_RANGE = (5, 30)    # 33k-200k msgs/sec during bursts

# Size deception (spoofing):
ABUSE_LARGE_SIZE_PROB = 0.40           # 40% are large "spoof" orders
ABUSE_LARGE_SIZE_MULT = 15             # 15x normal size (very large)
ABUSE_TINY_FILL_FRAC_RANGE = (0.0005, 0.015)  # 0.05%-1.5% executed

# Price behavior for abuse:
ABUSE_PRICE_STD_TICKS = 1.5            # Tighter price range (stays near best bid/ask)

# ============================================================================
# TIMING PARAMETERS
# ============================================================================
START_TIME = datetime(2026, 1, 18, 9, 30, 0)  # Market open

# Normal orders start at market open
# Abuse orders start slightly after (to blend in initially)
ABUSE_START_OFFSET_SEC = 30            # Start 30 seconds after normal orders

# ============================================================================
# GENERATE DATA
# ============================================================================

# Generate normal trading data (realistic market participants)
df_normal = generate_orders(
    n_orders=N_NORMAL_ORDERS, 
    start_order_id=10, 
    start_time=START_TIME,
    base_price=BASE_PRICE,              # Use realistic base price
    price_std=PRICE_STD,                # Use realistic price volatility
    n_traders=N_NORMAL_TRADERS
)

# Generate abuse data (suspicious trading patterns)
df_abuse = generate_abuse_orders(
    n_orders=N_ABUSE_ORDERS, 
    start_order_id=2,
    start_time=START_TIME + timedelta(seconds=ABUSE_START_OFFSET_SEC),
    
    # Market microstructure
    base_price=BASE_PRICE,
    tick_size=TICK_SIZE,
    lot_size=LOT_SIZE,
    price_std_ticks=ABUSE_PRICE_STD_TICKS,
    n_traders=N_ABUSE_TRADERS,
    
    # Abuse profile
    cancel_ratio_target=ABUSE_CANCEL_RATIO_TARGET,
    min_lifetime_us=ABUSE_MIN_LIFETIME_US,
    max_lifetime_us=ABUSE_MAX_LIFETIME_US,
    
    # Bursty message dynamics (quote stuffing)
    baseline_iat_us=ABUSE_BASELINE_IAT_US,
    burst_prob=ABUSE_BURST_PROB,
    burst_len_range=ABUSE_BURST_LEN_RANGE,
    burst_iat_us_range=ABUSE_BURST_IAT_US_RANGE,
    
    # Size deception (spoofing)
    large_size_prob=ABUSE_LARGE_SIZE_PROB,
    large_size_mult=ABUSE_LARGE_SIZE_MULT,
    tiny_fill_frac_range=ABUSE_TINY_FILL_FRAC_RANGE,
)

In [79]:
# Merge both datasets
df_combined = pd.concat([df_normal, df_abuse], ignore_index=True)
df_combined = df_combined.sort_values("timestamp").reset_index(drop=True)

print(f"Normal orders: {len(df_normal)} events")
print(f"Abuse orders: {len(df_abuse)} events")
print(f"Combined: {len(df_combined)} events")
print(f"\nCombined data preview:")

df_combined.head(20)

Normal orders: 275086 events
Abuse orders: 61041 events
Combined: 336127 events

Combined data preview:


,order_id,trader_id,side,price,quantity,remaining,event_type,order_type,timestamp,status
0,11,T579,Sell,101.09,200,200,NEW,Limit,2026-01-18 09:30:00.000,Open
1,11,T579,Sell,101.09,200,100,TRADE,Limit,2026-01-18 09:30:00.111,Partially filled
2,11,T579,Sell,101.09,200,0,TRADE,Limit,2026-01-18 09:30:00.311,Filled
3,12,T245,Sell,101.51,250,250,NEW,Limit,2026-01-18 09:30:00.502,Open
4,12,T245,Sell,101.51,250,90,TRADE,Limit,2026-01-18 09:30:00.586,Partially filled
5,12,T245,Sell,101.51,250,0,TRADE,Limit,2026-01-18 09:30:00.646,Filled
6,13,T661,Sell,101.47,250,250,NEW,Limit,2026-01-18 09:30:00.994,Open
7,13,T661,Sell,101.47,250,250,CANCEL,Limit,2026-01-18 09:30:01.182,Cancelled
8,14,T952,Buy,101.39,100,100,NEW,Limit,2026-01-18 09:30:01.478,Open
9,14,T952,Buy,101.39,100,31,TRADE,Limit,2026-01-18 09:30:01.612,Partially filled


In [80]:
df_normal

,order_id,trader_id,side,price,quantity,remaining,event_type,order_type,timestamp,status
0,11,T579,Sell,101.09,200,200,NEW,Limit,2026-01-18 09:30:00.000,Open
1,11,T579,Sell,101.09,200,100,TRADE,Limit,2026-01-18 09:30:00.111,Partially filled
2,11,T579,Sell,101.09,200,0,TRADE,Limit,2026-01-18 09:30:00.311,Filled
3,12,T245,Sell,101.51,250,250,NEW,Limit,2026-01-18 09:30:00.502,Open
4,12,T245,Sell,101.51,250,90,TRADE,Limit,2026-01-18 09:30:00.586,Partially filled
...,...,...,...,...,...,...,...,...,...,...
275081,100009,T233,Sell,101.45,250,168,TRADE,Limit,2026-01-19 02:36:14.854,Partially filled
275082,100009,T233,Sell,101.45,250,0,TRADE,Limit,2026-01-19 02:36:15.001,Filled
275083,100010,T944,Buy,101.42,300,300,NEW,Limit,2026-01-19 02:36:15.285,Open
275084,100010,T944,Buy,101.42,300,156,TRADE,Limit,2026-01-19 02:36:15.453,Partially filled


In [81]:
df_combined

,order_id,trader_id,side,price,quantity,remaining,event_type,order_type,timestamp,status
0,11,T579,Sell,101.09,200,200,NEW,Limit,2026-01-18 09:30:00.000,Open
1,11,T579,Sell,101.09,200,100,TRADE,Limit,2026-01-18 09:30:00.111,Partially filled
2,11,T579,Sell,101.09,200,0,TRADE,Limit,2026-01-18 09:30:00.311,Filled
3,12,T245,Sell,101.51,250,250,NEW,Limit,2026-01-18 09:30:00.502,Open
4,12,T245,Sell,101.51,250,90,TRADE,Limit,2026-01-18 09:30:00.586,Partially filled
...,...,...,...,...,...,...,...,...,...,...
336122,100009,T233,Sell,101.45,250,168,TRADE,Limit,2026-01-19 02:36:14.854,Partially filled
336123,100009,T233,Sell,101.45,250,0,TRADE,Limit,2026-01-19 02:36:15.001,Filled
336124,100010,T944,Buy,101.42,300,300,NEW,Limit,2026-01-19 02:36:15.285,Open
336125,100010,T944,Buy,101.42,300,156,TRADE,Limit,2026-01-19 02:36:15.453,Partially filled


# Anonimize

In [82]:
df = df_combined.copy()

df = df.sample(frac=1, random_state=420).reset_index(drop=True)


# 1) Anonymize trader_id (stable mapping)
trader_map = {t: f"TRADER_{i:03d}" for i, t in enumerate(df["trader_id"].dropna().unique(), start=1)}
df["trader_id"] = df["trader_id"].map(trader_map)

# 2) Anonymize order_id (stable mapping)
order_map = {oid: i for i, oid in enumerate(df["order_id"].dropna().unique(), start=1)}
df["order_id"] = df["order_id"].map(order_map)

df_anonymized = df

# If you also want to REMOVE any hint in the text (e.g., "ABUSE_T03" pattern),
# the mapping above already does it, since it replaces the whole trader_id.


In [83]:
df

,order_id,trader_id,side,price,quantity,remaining,event_type,order_type,timestamp,status
0,1,TRADER_001,Sell,101.63,150,99,TRADE,Limit,2026-01-19 01:06:57.804,Partially filled
1,2,TRADER_002,Buy,101.14,300,0,TRADE,Limit,2026-01-19 00:21:47.758,Filled
2,3,TRADER_003,Buy,101.16,300,0,TRADE,Limit,2026-01-18 20:12:07.899,Filled
3,4,TRADER_004,Sell,101.38,300,108,TRADE,Limit,2026-01-18 17:27:21.554,Partially filled
4,5,TRADER_005,Buy,101.38,200,115,TRADE,Limit,2026-01-18 14:11:56.834,Partially filled
...,...,...,...,...,...,...,...,...,...,...
336122,94547,TRADER_154,Sell,101.55,250,0,TRADE,Limit,2026-01-18 16:46:47.959,Filled
336123,15966,TRADER_460,Sell,101.44,100,0,TRADE,Limit,2026-01-18 09:34:30.797,Filled
336124,26333,TRADER_016,Buy,101.51,200,99,CANCEL,Limit,2026-01-18 15:10:00.994,Cancelled
336125,81098,TRADER_628,Sell,101.33,250,250,NEW,Limit,2026-01-18 13:06:52.917,Open


In [84]:
display(df_anonymized)

,order_id,trader_id,side,price,quantity,remaining,event_type,order_type,timestamp,status
0,1,TRADER_001,Sell,101.63,150,99,TRADE,Limit,2026-01-19 01:06:57.804,Partially filled
1,2,TRADER_002,Buy,101.14,300,0,TRADE,Limit,2026-01-19 00:21:47.758,Filled
2,3,TRADER_003,Buy,101.16,300,0,TRADE,Limit,2026-01-18 20:12:07.899,Filled
3,4,TRADER_004,Sell,101.38,300,108,TRADE,Limit,2026-01-18 17:27:21.554,Partially filled
4,5,TRADER_005,Buy,101.38,200,115,TRADE,Limit,2026-01-18 14:11:56.834,Partially filled
...,...,...,...,...,...,...,...,...,...,...
336122,94547,TRADER_154,Sell,101.55,250,0,TRADE,Limit,2026-01-18 16:46:47.959,Filled
336123,15966,TRADER_460,Sell,101.44,100,0,TRADE,Limit,2026-01-18 09:34:30.797,Filled
336124,26333,TRADER_016,Buy,101.51,200,99,CANCEL,Limit,2026-01-18 15:10:00.994,Cancelled
336125,81098,TRADER_628,Sell,101.33,250,250,NEW,Limit,2026-01-18 13:06:52.917,Open


In [85]:
df_combined.size

3361270

## Memory Usage Check

Check the memory footprint of the combined dataset.

In [86]:
# Check memory usage of df_combined
print("="*60)
print("DATAFRAME MEMORY USAGE")
print("="*60)

# Get detailed memory usage (deep=True includes object data)
memory_bytes = df_combined.memory_usage(deep=True).sum()
memory_kb = memory_bytes / 1024
memory_mb = memory_bytes / (1024 * 1024)

print(f"\nTotal memory usage: {memory_bytes:,} bytes")
print(f"                    {memory_kb:.2f} KB")
print(f"                    {memory_mb:.4f} MB")

print(f"\nDataFrame shape: {df_combined.shape} (rows × columns)")
print(f"DataFrame size:  {df_combined.size:,} (total elements = rows × columns)")

print(f"\nMemory usage by column:")
mem_by_col = df_combined.memory_usage(deep=True)
for col, mem in mem_by_col.items():
    if col != 'Index':  # Skip index column
        mem_kb = mem / 1024
        print(f"  {col:15s}: {mem:10,} bytes ({mem_kb:7.2f} KB)")

# Also show as return value
memory_bytes

DATAFRAME MEMORY USAGE

Total memory usage: 104,533,814 bytes
                    102083.80 KB
                    99.6912 MB

DataFrame shape: (336127, 10) (rows × columns)
DataFrame size:  3,361,270 (total elements = rows × columns)

Memory usage by column:
  order_id       :  2,689,016 bytes (2625.99 KB)
  trader_id      : 18,092,622 bytes (17668.58 KB)
  side           : 17,647,361 bytes (17233.75 KB)
  price          :  2,689,016 bytes (2625.99 KB)
  quantity       :  2,689,016 bytes (2625.99 KB)
  remaining      :  2,689,016 bytes (2625.99 KB)
  event_type     : 17,970,520 bytes (17549.34 KB)
  order_type     : 18,150,858 bytes (17725.45 KB)
  timestamp      :  2,689,016 bytes (2625.99 KB)
  status         : 19,227,241 bytes (18776.60 KB)


np.int64(104533814)

## Data Summary

View summary statistics and data quality checks.

In [87]:
# Data summary
print("="*60)
print("ORDER DATA SUMMARY")
print("="*60)
print(f"\nTotal events: {len(df)}")
print(f"Total orders: {df['order_id'].nunique()}")
print(f"\nTime span:")
print(f"  Start: {df['timestamp'].min()}")
print(f"  End: {df['timestamp'].max()}")
print(f"  Duration: {df['timestamp'].max() - df['timestamp'].min()}")

print(f"\nEvent types:")
print(df['event_type'].value_counts())

print(f"\nUnique traders: {df['trader_id'].nunique()}")
print(f"Unique order IDs: {df['order_id'].nunique()}")

print(f"\nSide distribution:")
print(df['side'].value_counts())

ORDER DATA SUMMARY

Total events: 336127
Total orders: 100008

Time span:
  Start: 2026-01-18 09:30:00
  End: 2026-01-19 02:36:15.708000
  Duration: 0 days 17:06:15.708000

Event types:
event_type
NEW       130000
TRADE     126465
CANCEL     79662
Name: count, dtype: int64

Unique traders: 1008
Unique order IDs: 100008

Side distribution:
side
Sell    168757
Buy     167370
Name: count, dtype: int64


## Market Abuse Metrics Analysis

Calculate and compare abuse metrics between normal and abuse data patterns.

In [88]:
def calculate_abuse_metrics(df, label="Dataset"):
    """
    Calculate market abuse detection metrics for a dataset.
    
    Returns:
    --------
    dict with metrics including cancel_ratio, avg_lifetime, msg_rate, size_ratio
    """
    # Calculate cancel ratio
    event_counts = df['event_type'].value_counts()
    n_adds = event_counts.get('NEW', 0)
    n_cancels = event_counts.get('CANCEL', 0)
    n_trades = event_counts.get('TRADE', 0)
    
    total_events = n_adds + n_cancels + n_trades
    cancel_ratio = n_cancels / total_events if total_events > 0 else 0
    
    # Calculate average order lifetime
    lifetimes = []
    for order_id in df['order_id'].unique():
        order_events = df[df['order_id'] == order_id].sort_values('timestamp')
        if len(order_events) > 1:
            first_time = order_events.iloc[0]['timestamp']
            last_time = order_events.iloc[-1]['timestamp']
            lifetime = (last_time - first_time).total_seconds() * 1000000  # microseconds
            lifetimes.append(lifetime)
    
    avg_lifetime_us = np.mean(lifetimes) if lifetimes else 0
    median_lifetime_us = np.median(lifetimes) if lifetimes else 0
    
    # Calculate message rate
    time_span = (df['timestamp'].max() - df['timestamp'].min()).total_seconds()
    msg_rate = len(df) / time_span if time_span > 0 else 0
    
    # Calculate size ratio (executed volume / posted volume)
    posted_volume = df[df['event_type'] == 'NEW']['quantity'].sum()
    
    # Calculate executed volume by tracking each order's filled quantity
    executed_volume = 0
    for order_id in df['order_id'].unique():
        order_events = df[df['order_id'] == order_id].sort_values('timestamp')
        new_event = order_events[order_events['event_type'] == 'NEW']
        if len(new_event) > 0:
            initial_quantity = new_event.iloc[0]['quantity']
            # Find final remaining quantity from last event
            last_event = order_events.iloc[-1]
            final_remaining = last_event['remaining']
            executed_volume += initial_quantity - final_remaining
    
    size_ratio = executed_volume / posted_volume if posted_volume > 0 else 0
    
    return {
        'label': label,
        'cancel_ratio': cancel_ratio,
        'avg_lifetime_us': avg_lifetime_us,
        'median_lifetime_us': median_lifetime_us,
        'msg_rate_per_sec': msg_rate,
        'size_ratio': size_ratio,
        'n_adds': n_adds,
        'n_cancels': n_cancels,
        'n_trades': n_trades,
        'posted_volume': posted_volume,
        'executed_volume': executed_volume
    }

# Calculate metrics for normal, abuse, and combined data
metrics_normal = calculate_abuse_metrics(df_normal, "Normal Data")
metrics_abuse = calculate_abuse_metrics(df_abuse, "Abuse Data")
metrics_combined = calculate_abuse_metrics(df_combined, "Combined Data")

# Display comparison
print("="*80)
print("MARKET ABUSE METRICS COMPARISON")
print("="*80)

for metrics in [metrics_normal, metrics_abuse, metrics_combined]:
    print(f"\n{metrics['label']}:")
    print(f"  Cancel Ratio:        {metrics['cancel_ratio']:.3f} ({metrics['cancel_ratio']*100:.1f}%)")
    print(f"  Avg Lifetime:        {metrics['avg_lifetime_us']:.2f} microseconds ({metrics['avg_lifetime_us']/1000:.2f} ms)")
    print(f"  Median Lifetime:     {metrics['median_lifetime_us']:.2f} microseconds ({metrics['median_lifetime_us']/1000:.2f} ms)")
    print(f"  Message Rate:        {metrics['msg_rate_per_sec']:.2f} messages/second")
    print(f"  Size Ratio:          {metrics['size_ratio']:.4f} ({metrics['size_ratio']*100:.2f}%)")
    print(f"  Posted Volume:       {metrics['posted_volume']:,.0f}")
    print(f"  Executed Volume:     {metrics['executed_volume']:,.0f}")
    print(f"  Events: NEW={metrics['n_adds']}, CANCEL={metrics['n_cancels']}, TRADE={metrics['n_trades']}")

print("\n" + "="*80)
print("INTERPRETATION:")
print("  Cancel Ratio: Higher = suspicious (abuse typically > 0.7)")
print("  Lifetime: Lower = suspicious (abuse typically < 1ms)")
print("  Message Rate: Higher = suspicious (abuse creates bursty activity)")
print("  Size Ratio: Lower = suspicious (abuse shows fake liquidity)")
print("="*80)

# Export

In [89]:
# Export df_combined to Parquet only (preferred for performance and size)
parquet_filename = "order_data_combined.parquet"
df_anonymized.to_parquet(parquet_filename, index=False)